In [ ]:
# ============================================================
# 00_SETUP.py  —  ATLAS GO Prediction Pipeline
# Run ONCE at the start to create folders, config, and src files
# ============================================================


In [7]:
## ── 1. Mount Google Drive ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
## ── 2. Create folder structure ──────────────────────────────
import os, yaml

Project_Root = '/content/drive/MyDrive/atlas-go-dynamicsV2'

folders = [
    f"{Project_Root}/data/raw",
    f"{Project_Root}/data/processed",
    f"{Project_Root}/results/tables",
    f"{Project_Root}/results/figures",
    f"{Project_Root}/configs",
    f"{Project_Root}/src",
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print(f" Folder structure created: {Project_Root}")


 Folder structure created: /content/drive/MyDrive/atlas-go-dynamicsV2


In [11]:
## ── 3. Write config.yaml ────────────────────────────────────
config_content = """\
paths:
  project_root: "/content/drive/MyDrive/atlas-go-dynamicsV2"
  data_raw: "/content/drive/MyDrive/atlas-go-dynamicsV2/data/raw"
  data_processed: "/content/drive/MyDrive/atlas-go-dynamicsV2/data/processed"
  results: "/content/drive/MyDrive/atlas-go-dynamicsV2/results/tables"
  figures: "/content/drive/MyDrive/atlas-go-dynamicsV2/results/figures"

training:
  max_epochs: 100
  optimizer: "adamw"
  early_stopping_patience: 10
  gradient_clip_norm: 1.0
  focal_loss_alpha: 0.25
  focal_loss_gamma: 2.0
  cosine_annealing_tmax: 50
  mlp_batch_size: 256
  mlp_learning_rate: 3.0e-4
  mlp_weight_decay: 1.0e-4
  cnn_batch_size: 256
  cnn_learning_rate: 3.0e-4
  cnn_weight_decay: 1.0e-4
  transformer_batch_size: 128
  transformer_learning_rate: 2.0e-4
  transformer_weight_decay: 1.0e-5
  transformer_d_model: 512
  transformer_num_layers: 3

random_seed: 42

data:
  go_namespace_filter: ["MFO", "BPO", "CCO"]
  min_proteins_per_go: 5
  train_ratio: 0.80
  val_ratio: 0.10
  test_ratio: 0.10
  esm2_max_seq_len: 1000
  static_features: ["α%", "β%", "Coil%", "Len."]
  dynamic_features: ["Avg. RMSF", "Avg. gyr.", "Div. SE", "Div. MM"]
"""

cfg_path = f"{Project_Root}/configs/config.yaml"
with open(cfg_path, "w") as f:
    f.write(config_content)
print(" config.yaml written!")

 config.yaml written!


In [12]:
## ── 4. Verify config loads correctly ────────────────────────
with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

print("\n" + "="*50)
print("  CONFIG VERIFIED")
print("="*50)
print(f"  Seed:              {cfg['random_seed']}")
print(f"  Min proteins/GO:   {cfg['data']['min_proteins_per_go']}")
print(f"  Split:             {cfg['data']['train_ratio']}/{cfg['data']['val_ratio']}/{cfg['data']['test_ratio']}")
print(f"  ESM2 max seq len:  {cfg['data']['esm2_max_seq_len']}")
print(f"  MLP  lr/bs:        {cfg['training']['mlp_learning_rate']} / {cfg['training']['mlp_batch_size']}")
print(f"  CNN  lr/bs:        {cfg['training']['cnn_learning_rate']} / {cfg['training']['cnn_batch_size']}")
print(f"  Transformer lr/bs: {cfg['training']['transformer_learning_rate']} / {cfg['training']['transformer_batch_size']}")
print(f"  Max epochs:        {cfg['training']['max_epochs']}")
print(f"  Early stop:        {cfg['training']['early_stopping_patience']} epochs")

## ── 5. Create empty src files ───────────────────────────────
src_files = {
    "__init__.py":    "# ATLAS GO Prediction Pipeline\n",
    "data_loading.py":"# Data loading, cleaning, GO parsing, splits\n",
    "features.py":    "# ProtBERT / ESM2 embeddings + S/D normalization\n",
    "models.py":      "# MLP, CNN1D, TransformerGO architectures + FocalLoss\n",
    "train_main.py":  "# Training loop — reads all hyperparams from config.yaml\n",
    "metrics.py":     "# CAFA metrics: Fmax + AUPRC for Overall/MFO/BPO/CCO\n",
}

src_path = f"{Project_Root}/src"
for filename, content in src_files.items():
    with open(f"{src_path}/{filename}", "w") as f:
        f.write(content)

print("\n src/ files created:")
for filename in src_files:
    print(f"   src/{filename}")

print("\n SETUP COMPLETE — run 01_data_preparation.py next")


  CONFIG VERIFIED
  Seed:              42
  Min proteins/GO:   5
  Split:             0.8/0.1/0.1
  ESM2 max seq len:  1000
  MLP  lr/bs:        0.0003 / 256
  CNN  lr/bs:        0.0003 / 256
  Transformer lr/bs: 0.0002 / 128
  Max epochs:        100
  Early stop:        10 epochs

 src/ files created:
   src/__init__.py
   src/data_loading.py
   src/features.py
   src/models.py
   src/train_main.py
   src/metrics.py

 SETUP COMPLETE — run 01_data_preparation.py next
